# Planners

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/08_planners.ipynb)

One problem, the planner family: each planner solves it into a `PlanningSolution`, and its compiled face is what an MPC runs.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Problem → planner → solution

A planner takes a `PlanningProblem` and its own solver options; `solve()` returns a
`PlanningSolution`, the same record for every planner:

- `policy`: the answer as a `System`, a block that wires into a diagram like any controller;
- `trajectory`: the planned `Trajectory`, when the method produces one;
- `solver`: the record of the solve (success, cost, iterations, the method's own evidence).

The task: swing a pendulum from rest at the bottom to the upright in five seconds, within
the torque limit. `TrajectoryOptimizationPlanner` transcribes it into a `MathematicalProgram`
(here by direct collocation on 20 knots) and solves it; its policy is the open-loop input
plan, a `TrajectorySource`.

In [ ]:
import numpy as np

from minilink import Pendulum, PlanningProblem, QuadraticCost
from minilink.planning import TrajectoryOptimizationPlanner

plant = Pendulum()
UP = np.array([np.pi, 0.0])
cost = QuadraticCost.from_system(plant, xbar=UP, Q=np.eye(2), R=0.1 * np.eye(1))
problem = PlanningProblem(
    plant, x_start=np.zeros(2), cost=cost, U=plant.input_box, Xf=UP, tf=5.0
)

trajopt = TrajectoryOptimizationPlanner(
    problem, n_steps=20, transcription="direct_collocation", backend="jax"
)
plan = trajopt.solve()
print(plan)

In [ ]:
plant.plot_trajectory(plan.trajectory)

## 2. The planner family on the same task

Each planner reads the form of the problem its method needs, derived with `replace`. LQR
and value iteration solve the infinite-horizon regulation to the upright (value iteration
with a discount, so it converges on a grid); each returns a feedback policy $u = \pi(x)$.

- `LQRPlanner` linearizes at the target and solves the Riccati equation: a local law, which
  ignores the torque limit (its record says so).
- `DynamicProgrammingPlanner` runs value iteration on a state grid with the torque limit
  enforced: a global law, a lookup table interpolated on the grid.

`compare(name=solution, ...)` lines up the solutions' records.

In [ ]:
from minilink import DynamicProgrammingPlanner, LQRPlanner, compare

regulation = problem.replace(tf=None, Xf=None, discount_rate=0.2)

lqr_solution = LQRPlanner(regulation).solve()
value_iteration = DynamicProgrammingPlanner(
    regulation, x_grid=(81, 81), u_grid=(11,), dt=0.05, tol=0.1
)
vi_solution = value_iteration.solve()

print(compare(TO=plan, LQR=lqr_solution, VI=vi_solution))

A policy is a `System`, so closing the loop is the `@` of `01_systems`. The value-iteration
law swings the pendulum up from the bottom, pumping energy within the torque limit:

In [ ]:
vi_loop = vi_solution.policy @ plant
plant.x0 = np.zeros(2)
vi_traj = vi_loop.compute_trajectory(tf=8.0)

In [ ]:
vi_loop.plot_trajectory(vi_traj)

## 3. The compiled face of a planner

`planner.compile()` gives the planner's pure face, `pv`: its solve written as functions of a
solution vector `z` and the planner's parameter tree, with no hidden last solution.

| Method | Meaning |
| --- | --- |
| `pv.n_z`, `pv.params` | the size of `z`; the tree the plan depends on (`x_start`, `t_start`, the plant's and the cost's numbers, the solver's) |
| `pv.initial(params)` | the default guess, in `z` form |
| `pv.solve(params, z0)` | `(z, status)`: one solve from a guess |
| `pv.shift(z, t)` | the plan re-based to time `t`: the warm start of the next solve |
| `pv.command(z, x, t)` | the input the plan gives at time `t` |

The planner below is the one an MPC runs at every tick: a one-second horizon, single
shooting on ten knots, and a fixed count of Adam steps, so a tick is a pure function that
traces under JAX. It starts from a 0.5 rad tilt below the upright.

In [ ]:
x_tilt = np.array([np.pi - 0.5, 0.0])
task = PlanningProblem(plant, x_start=x_tilt, cost=cost, U=plant.input_box, tf=4.0)

short_planner = TrajectoryOptimizationPlanner(
    task.replace(tf=1.0),
    n_steps=10,
    transcription="single_shooting",
    backend="jax",
    method="adam",
    iterations=30,
    optimizer_options={"learning_rate": 0.2},
)
pv = short_planner.compile()
pv.n_z, list(pv.params)

One receding-horizon step by hand: solve from the guess, read the command at $t = 0$, shift
the plan by one period, and solve again from the new measured state, warm-started.

In [ ]:
z, status = pv.solve(z0=pv.initial())
u_now = pv.command(z, x_tilt, 0.0)

z_warm = pv.shift(z, 0.1)
x_measured = np.array([np.pi - 0.3, 0.0])
z_next, status_next = pv.solve(
    params={"x_start": x_measured, "t_start": 0.1}, z0=z_warm
)
status, u_now, status_next

## 4. The MPC is built on the face

`ModelPredictiveController(planner)` is that loop as a block: at each tick it binds the
measured state and the time into the tree, solves warm-started from the shifted plan, and
holds the command. Put it on a clock and it is a digital controller like any other:
`(mpc % dt) @ plant` simulates it, and `problem.compile(mpc % dt)` scores it on the task, in
one compiled scan.

In [ ]:
from minilink.control import ModelPredictiveController

mpc = ModelPredictiveController(short_planner)
pe = task.compile(mpc % 0.1, substeps=1, backend="jax")
print(pe.evaluate())

mpc_traj = pe.trajectory()

In [ ]:
pe.loop.plot_trajectory(mpc_traj)

## Summary: the pipeline

System (`01`) → parameters (`02`) → sets, fields, distributions (`03`) → costs (`04`) →
`PlanningProblem` (`05`) → `problem.compile` and `evaluate` (`06`) → `MathematicalProgram`
and `Optimizer` (`07`) → planners, their `PlanningSolution`, and their compiled face (`08`).

The advanced layer goes deeper into each tool: discrete and hybrid loops, optimal control
compared, MPC and MPPI, reinforcement learning, and differentiable loops.